# CDC downloads — getting the raw files back

The two file-delivered CDC sources have no API to re-query on demand, so the
raw files are fetched once and parsed from disk. This notebook is how they are
fetched, and its saved output is the record that they were. What is *not*
regenerable is knowing how — hence a notebook rather than a script.

## Where the data goes

Nothing here reads local input. The inputs are remote; everything below is
written under `notebooks/cdc/data/`, which is **gitignored and regenerable**.

| | written to | what it is |
| --- | --- | --- |
| NVSR national | `data/nvsr/us/<year>/Table*.xlsx` | 120 life tables, 2018–2024 |
| NVSR state | `data/nvsr/state/<year>/{ST}[1-3].xlsx` | 816 workbooks, 2018–2022 |
| WONDER | `data/wonder/<concept>_<D76\|D158>.json` | 9 concepts × 2 database vintages |
| WONDER | `data/wonder/_download_summary.json` | **the only record of which codes WONDER refused** |

Sources: NVSR from `ftp.cdc.gov/pub/Health_Statistics/NCHS/Publications/NVSR/`,
by volume-number directory — the year → volume mapping lives in
`utils/nvsr_series.py` and exists nowhere programmatic. WONDER from its
XML-POST API, throttled to roughly one query per two minutes.

**Downstream, and not produced here:** `utils/nvsr_series.py` and
`utils/wonder_series.py` parse these into `data/timeseries/*.jsonl`, which
`db_import/load_timeseries.py` loads into Postgres. Re-running this notebook
does not rebuild those — see `catalog.ipynb`.

## Running it

**The default is a no-op.** Both fetchers skip anything already on disk, so a
plain Run All transfers nothing when the tree is complete. It still lists each
volume directory to decide what it wants — 12 requests to `ftp.cdc.gov` — and
makes no WONDER queries at all.

To actually re-fetch, pass `refresh=True`. **Deleting the old files is not
required** and is the worse option, since it destroys the tree before knowing
whether the re-fetch will succeed:

```python
fetch.fetch_national(refresh=True)      # or fetch_state / fetch_all
fetch.fetch_wonder(refresh=True)        # ~40 min, throttled
```

A refreshed NVSR fetch reports `revised` — the files whose bytes changed. That
matters because **NCHS replaces published workbooks in place** when it corrects
them; the 2021 volume was re-issued in December 2023 under the same name. A
re-fetch without that check would report "downloaded: 153" either way.

Costs, cold: NVSR about 15 minutes for 936 workbooks at 1s pacing; WONDER about
40 minutes for 9 concepts across both vintages. `ftp.cdc.gov` bot-filters on
*rate* rather than per request — a pull at ~3 files/s served 400 workbooks and
then stalled every read — so the pacing in `utils/fetch.py` is load-bearing.

In [1]:
import sys
sys.path.append("../")     # notebooks/<source>/ helpers
sys.path.append("../../")  # repo root

from utils import fetch
from utils import nvsr_series as ns
from utils import wonder_codes

## NVSR — 936 Excel workbooks off FTP

Everything here is idempotent: a workbook already on disk is skipped, so
re-running resumes rather than re-downloads. That matters more than usual —
`ftp.cdc.gov`'s bot filter is **rate-based**, and a pull at ~3 files/s served
400 workbooks before stalling every subsequent read to a timeout. `fetch` uses
`curl_cffi` with a browser fingerprint and a 1s pause; don't lower it.

Two naming quirks the fetcher absorbs, so the local tree is uniform whatever
the remote year called things:

| | quirk |
| --- | --- |
| 2018 state (`70-01`) | files are `Alabama-1-Total.xlsx`, not `AL1.xlsx` |
| 2020 national (`71-01`) | lowercase `table01.xlsx` |
| every state year | `{ST}4` is **standard errors**, not a life table — skipped |

Expect ~15 minutes on a cold run, seconds when everything is present.

In [2]:
national = fetch.fetch_national()

  2018 (69-12): {'wanted': 12, 'downloaded': 0, 'revised': [], 'on_disk': 12}


  2019 (70-19): {'wanted': 18, 'downloaded': 0, 'revised': [], 'on_disk': 18}


  2020 (71-01): {'wanted': 18, 'downloaded': 0, 'revised': [], 'on_disk': 18}
  2021 (72-12): {'wanted': 18, 'downloaded': 0, 'revised': [], 'on_disk': 18}


  2022 (74-02): {'wanted': 18, 'downloaded': 0, 'revised': [], 'on_disk': 18}
  2023 (74-06): {'wanted': 18, 'downloaded': 0, 'revised': [], 'on_disk': 18}


  2024 (75-05): {'wanted': 18, 'downloaded': 0, 'revised': [], 'on_disk': 18}


In [3]:
state = fetch.fetch_state()

  2018 (70-01): {'wanted': 153, 'downloaded': 0, 'revised': [], 'on_disk': 153}


  2019 (70-18): {'wanted': 153, 'downloaded': 0, 'revised': [], 'on_disk': 153}


  2020 (71-02): {'wanted': 153, 'downloaded': 0, 'revised': [], 'on_disk': 153}


  2021 (73-07): {'wanted': 153, 'downloaded': 0, 'revised': [], 'on_disk': 153}


  2022 (74-12): {'wanted': 153, 'downloaded': 0, 'revised': [], 'on_disk': 204}


## Check the parse before trusting the files

A truncated or substituted workbook still parses — it just returns a wrong
number. Both guards run over everything on disk and raise rather than
returning quietly-wrong values.

`verify_national` compares age-0 life expectancy against the published
figures. `verify_state` has no published table to check against (51
jurisdictions a year is too many to transcribe), so it leans on three things
a misread cannot satisfy: `female > both > male`, `e0` within 60–95, and the
national value falling inside the state range.

In [4]:
print("national e0 vs published:")
for year, value in sorted(ns.verify_national().items()):
    print(f"  {year}  {value:.4f}  -> {round(value, 1)}  (published {ns.PUBLISHED_E0[year]})")

print("\nstate jurisdictions per year:", ns.verify_state())

national e0 vs published:
  2018  78.7375  -> 78.7  (published 78.7)
  2019  78.8482  -> 78.8  (published 78.8)
  2020  76.9950  -> 77.0  (published 77.0)
  2021  76.3702  -> 76.4  (published 76.4)
  2022  77.4569  -> 77.5  (published 77.5)
  2023  78.4214  -> 78.4  (published 78.4)
  2024  78.9710  -> 79.0  (published 79.0)



state jurisdictions per year: {2018: 51, 2019: 51, 2020: 51, 2021: 51, 2022: 51}


## WONDER — 9 concepts × 2 database vintages

The concepts are ICD-10 code sets, and those sets were lost: the original pull
ran from a script that is gone, and the cached JSON holds results, not
queries. `wonder_codes.py` reconstructs them, and they are **verified against
the original pull** — query WONDER, compare deaths year by year. Seven of the
nine reproduce it exactly across 22 years.

Two kinds of code get refused, and `fetch_wonder` drops what WONDER names and
retries, recording the deviation:

| refused | why | affects |
| --- | --- | --- |
| `*U01`, `*U02`, `*U03`, `*U01.4` | NCHS pseudo-codes for terrorism reclassification — part of the published definitions, rejected by the finder | suicide, homicide, firearm, despair |
| `I03`, `I04`, `I23`, `I29`, `I32`, `I39`, `I41`, `I43` | gaps in the diseases-of-heart range; they do not exist in ICD-10 | cardiometabolic |

Those series therefore omit terrorism-reclassified deaths — a handful a year,
but a real departure from the published definition, so `dropped_codes` is
carried into each series' metadata rather than left in a commit message.

**Cached by default.** At roughly one query per two minutes, a full refresh of
all nine across both vintages is about forty minutes. Pass `refresh=True`
deliberately.

In [5]:
for concept, codes in sorted(wonder_codes.CODE_SETS.items()):
    rebuilt, recorded = wonder_codes.check_counts()[concept]
    flag = "ok" if rebuilt == recorded else "DIFFERS"
    print(f"  {concept:20s} {rebuilt:4d} codes  ({flag} vs the original pull)"
          f"   {wonder_codes.CITATIONS[concept]}")


  alcohol_induced        14 codes  (ok vs the original pull)   NCHS Data Brief 448 / WONDER UCD help
  cardiometabolic        44 codes  (ok vs the original pull)   NVSR 70-08 Table C p.10 (heart disease)
  chronic_liver           3 codes  (ok vs the original pull)   NVSR 70-08 Table C p.10
  despair_composite     160 codes  (ok vs the original pull)   union: alcohol_induced+drug_induced+suicide
  drug_induced          125 codes  (ok vs the original pull)   NVSR 70-08 p.74
  drug_overdose          16 codes  (ok vs the original pull)   NVSR 70-08 p.74 (subcategory)
  firearm                14 codes  (ok vs the original pull)   NVSR 70-08 p.75
  homicide               28 codes  (ok vs the original pull)   NVSR 70-08 p.11,40,43
  suicide                27 codes  (ok vs the original pull)   NVSR 70-08 Table C p.10


Cached, so this is a no-op unless a file is missing. The printed
`dropped_codes` is the record of what WONDER would not accept.

In [6]:
summary = await fetch.fetch_wonder()

for concept in sorted(summary):
    for database, info in sorted(summary[concept]["databases"].items()):
        dropped = info.get("dropped_codes") or []
        if dropped:
            print(f"  {concept:20s} {database:5s} dropped {dropped}")


  cardiometabolic      D158  dropped ['I03', 'I04', 'I23', 'I29', 'I32', 'I39', 'I41', 'I43']
  cardiometabolic      D76   dropped ['I03', 'I04', 'I23', 'I29', 'I32', 'I39', 'I41', 'I43']
  despair_composite    D158  dropped ['*U03']
  despair_composite    D76   dropped ['*U03']
  firearm              D158  dropped ['*U01.4']
  firearm              D76   dropped ['*U01.4']
  homicide             D158  dropped ['*U01', '*U02']
  homicide             D76   dropped ['*U01', '*U02']
  suicide              D158  dropped ['*U03']
  suicide              D76   dropped ['*U03']


### Re-pulling shows what NCHS changed

WONDER revises: death certificates get reclassified after publication. A
refresh compared against the tracked `data/timeseries/cdc_wonder.jsonl` is how
that becomes visible — re-running drug-induced today moves eight of its
twenty-two years by one or two deaths, in both directions. That is the reason
the normalized `.jsonl` is committed while the raw pulls are not.

In [7]:
from utils import wonder_series as ws

# compare the on-disk pull against what the loaded series holds
stored = {o["date"][:4]: o["value"]
          for o in ws.build_series("drug_induced")["observations"]}
cached = {str(r["year"]): r["age_adjusted_rate"]
          for r in wonder_codes.cached_series("drug_induced", "D76")}
same = sum(1 for y in cached if y in stored and float(stored[y]) == float(cached[y]))
print(f"  drug_induced: {same}/{len(cached)} D76 years agree between "
      f"the cached pull and the built series")


  drug_induced: 22/22 D76 years agree between the cached pull and the built series
